What should we do in this pyhton notebook code:-

Raw columns:

Column                      Meaning                               Usable for Rules?

Time       Seconds elapsed since first transaction in dataset    ✅ Convert to hour

V1–V28     PCA-anonymized features (real data hidden for privacy)⚠️ Limited

Amount     Transaction amount in euros                           ✅ Primary rule driver

Class      0 = legitimate, 1 = fraud (ground truth label)        ✅ Evaluate rules against this

The challenge: No account_id, no merchant, no location — all identifiers were removed. This is realistic — in real fraud 

work, raw data is often partially masked.Our job is to engineer what we can and build rules on available signals.

What Python will add:

transaction_id   → sequential ID for easy reference

hour_of_day      → Time % 86400 / 3600 (hour 0-23)

day_number       → Time // 86400 (day 1 or 2 — dataset spans ~2 days)

amount_log       → log(Amount + 1) for statistical work

account_id       → simulated from V1+V2 clustering (explained below)

In [52]:
import sys
import pandas as pd    #read CSV, manipulate data
import numpy as np     #math operations, random number generation
from sqlalchemy import create_engine, text   # connect Python to PostgreSQL and run SQL text queries
from sqlalchemy.types import Integer, BigInteger, Numeric, SmallInteger, VARCHAR, Float

try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

In [ ]:
# ── config ────────────────────────────────────────────────
# Database connection parameters
DB_USER ="postgres"
DB_PASSWORD ="****"
DB_HOST ="localhost"
DB_PORT ="5432"
DB_NAME ="fraud_detection"
CSV_PATH = r"J:\python\DA_project\SQL Projects\financial data fraud detection\data\creditcard.csv"
TABLE_NAME = "transactions"

In [54]:
np.random.seed(42)   # makes random number assignment reproducible. Every time we run the script we get the exact same salary values. 
                     #Without this, salaries would change on every run.Seed set to 42 so that random numbers will be identical on every run.
df=pd.read_csv(CSV_PATH)
print("Original shape:",df.shape)                   #(rows, columns)
print("Columns:",df.columns.tolist())               #all column names in a list
print("\nFraud Distribution:")                      
print(df['Class'].value_counts())                   #how many fraud (1) vs legit (0)
print(f"Fraud rate: {df['Class'].mean()*100:.4f}%")  #percentage of transactions that are fraudulent

Original shape: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Fraud Distribution:
Class
0    284315
1       492
Name: count, dtype: int64
Fraud rate: 0.1727%


Key observation: Only 492 out of 284,807 are fraud = 17.2749%. This is called a class imbalance problem. If our rule flags 1,000 transactions and only 10 are real fraud, that's still 99% false positives. This is why we measure precision and recall separately — not just accuracy. Also notice V1–V28 are just numbers (PCA-transformed), not readable features like "merchant name".

In [55]:
# ── 1. Add transaction_id ─────────────────────────────────
df.insert(0, 'transaction_id', range(1, len(df) + 1))  #inserts the column at position 0 (first column)
# The raw dataset has no transaction ID. We create one using range(1, len(df)+1) — a sequential number from 1 to 284,807.
#  Why needed? In SQL, we need a primary key to reference individual rows. Without this, we can't join tables back to a 
# specific transaction later.

In [56]:
# ── 2. Time-based features ────────────────────────────────
# Time column = seconds elapsed since first transaction
# Dataset spans ~48 hours (172792 seconds max)
df['hour_of_day'] = (df['Time'] % 86400/3600).astype(int)
df['day_number']=(df['Time'] // 86400 + 1).astype(int)    # day 1 or 2

# Is this an unusual hour? (midnight to 5am = high fraud period)
df['unusual_hour'] = df['hour_of_day'].apply(
    lambda h:1 if h in [0,1,2,3,4] else 0
)
print(df[['Time','hour_of_day','day_number','unusual_hour']].tail())

            Time  hour_of_day  day_number  unusual_hour
284802  172786.0           23           2             0
284803  172787.0           23           2             0
284804  172788.0           23           2             0
284805  172788.0           23           2             0
284806  172792.0           23           2             0


Why this matters for SQL: In SQL Query 3, we use is_unusual_hour = 1 as a fraud detection rule. This block proves the rule has signal — 27.2% of fraud happens in unusual hours vs only 10.2% of all transactions. That's a 2.7× fraud rate lift.

In [57]:
# ── 3. Amount-based features ──────────────────────────────
# What this block does: Builds two statistical columns from Amount
# log(Amount + 1) — the +1 prevents log(0) error
# np.log1p() is exactly log(x+1) but more numerically stable
df['amount_log'] = np.log1p(df['Amount'])

# Calculate global mean and std of Amount
amount_mean = df['Amount'].mean()
amount_std  = df['Amount'].std()

# Z-score: how many std deviations is this transaction from average?
# z = (value - mean) / std
# z > 3 = extremely high (top 0.13% of any normal distribution)
df['amount_zscore'] = (df['Amount'] - amount_mean) / amount_std

print("── Amount Statistics ──────────────────────")
print(f"Mean Amount   : ${amount_mean:.2f}")
print(f"Std Amount    : ${amount_std:.2f}")
print(f"Min Amount    : ${df['Amount'].min():.2f}")
print(f"Max Amount    : ${df['Amount'].max():.2f}")
print(f"3σ threshold  : ${amount_mean + 3*amount_std:.2f}")
print(f"Median Amount : ${df['Amount'].median():.2f}")
print()
print("── Z-Score Check ──────────────────────────")
# Define the boolean mask first — True where z-score is extreme
high_z_mask = df['amount_zscore'].abs() > 3
# Count how many transactions are flagged
high_z = high_z_mask.sum()
print(f"Transactions with |z| > 3 : {high_z}")

# Now use high_z_mask correctly — Class column still exists at this point
# Note: rename to is_fraud happens in Block 7, so here it's still 'Class'
print(f"Fraud within those        : {df[high_z_mask & (df['Class'] == 1)].shape[0]}")
print()
print(df[['Amount', 'amount_log', 'amount_zscore']].head(5))

── Amount Statistics ──────────────────────
Mean Amount   : $88.35
Std Amount    : $250.12
Min Amount    : $0.00
Max Amount    : $25691.16
3σ threshold  : $838.71
Median Amount : $22.00

── Z-Score Check ──────────────────────────
Transactions with |z| > 3 : 4076
Fraud within those        : 11

   Amount  amount_log  amount_zscore
0  149.62    5.014760       0.244964
1    2.69    1.305626      -0.342474
2  378.66    5.939276       1.160684
3  123.50    4.824306       0.140534
4   69.99    4.262539      -0.073403


**Understand the median vs mean gap**: Mean = $88, Median = $22. This huge gap means a few very large transactions are pulling the mean up. This is exactly why z-score fraud detection works — a $5,000 transaction is 19.6 standard deviations above the mean. No legitimate purchase pattern looks like that.

In [58]:
# ── 4. Simulate account_id ────────────────────────────────
# WHY pd.qcut instead of pd.cut:
# pd.cut()  = equal WIDTH bins  → fails when data has outliers (our case)
#             most bins empty, a few bins overloaded
# pd.qcut() = equal FREQUENCY bins → each bin gets same number of transactions
#             much more even account distribution

# q=500 → split V1 into 500 quantile-based buckets
# rank(method='first') → breaks ties in quantile edges (avoids duplicate edge error)
v1_bins = pd.qcut(df['V1'].rank(method='first'), q=500, labels=False)
v2_bins = pd.qcut(df['V2'].rank(method='first'), q=4,   labels=False)

df['account_id'] = (v1_bins.fillna(0) * 4 + v2_bins.fillna(0)).astype(int) + 1

print(f"Unique accounts    : {df['account_id'].nunique()}")
print(f"Avg txns/account   : {len(df)/df['account_id'].nunique():.2f}")
print(f"Max txns/account   : {df['account_id'].value_counts().max()}")
print(f"Min txns/account   : {df['account_id'].value_counts().min()}")
print(f"Account_id dtype   : {df['account_id'].dtype}")
print("\nTop 5 accounts by volume:")
print(df['account_id'].value_counts().head(5))
print("\nSample rows:")
print(df[['transaction_id','V1','V2','account_id','Amount','Class']].head(8))

Unique accounts    : 1952
Avg txns/account   : 145.91
Max txns/account   : 539
Min txns/account   : 1
Account_id dtype   : int64

Top 5 accounts by volume:
account_id
1997    539
1577    526
1573    520
1581    507
1993    503
Name: count, dtype: int64

Sample rows:
   transaction_id        V1        V2  account_id  Amount  Class
0               1 -1.359807 -0.072781         318  149.62      0
1               2  1.191857  0.266151        1355    2.69      0
2               3 -1.358354 -1.340163         317  378.66      0
3               4 -0.966272 -0.185226         478  123.50      0
4               5 -1.158233  0.877737         392   69.99      0
5               6 -0.425966  0.960523         804    3.67      0
6               7  1.229658  0.141004        1407    4.99      0
7               8 -0.644269  1.417964         668   40.80      0


 This account_id is synthetic — engineered from PCA components as a proxy. It is NOT real account data. In a real bank fraud system, We would have actual account_id in the raw data. The purpose here is to enable frequency-based SQL analysis (window functions over account + time)

In [59]:
# ── 5. Rename Class to is_fraud for clarity ───────────────
# Class is a poor column name for SQL — it's a reserved word in some databases and not self-explanatory. Renaming to is_fraud makes every SQL query readable:
# • WHERE is_fraud = 1 is instantly clear
# • WHERE Class = 1 is ambiguous
df.rename(columns={'Class':'is_fraud'}, inplace=True)
print('Class' in df.columns)
print('is_fraud'in df.columns)

False
True


In [60]:
# ── 6. Validation ─────────────────────────────────────────
# This is our last checkpoint before loading 284,807 rows into PostgreSQL. 
# Always validate at this stage — it's much easier to fix in Python than to reload from scratch after finding an issue in SQL.
# Check: shape, nulls in key columns, value ranges, dtypes, and a full column list.
print(f"\n📐 Shape     : {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\n📋 All columns:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2}. {col}")

print("\n🔍 Null check on key engineered columns:")
key_cols = ['transaction_id', 'hour_of_day', 'day_number',
            'unusual_hour', 'amount_log', 'amount_zscore', 'account_id']
print(df[key_cols].isnull().sum())

print("\n📊 Value ranges:")
print(f"   hour_of_day   : {df['hour_of_day'].min()} – {df['hour_of_day'].max()}")
print(f"   day_number    : {df['day_number'].min()} – {df['day_number'].max()}")
print(f"   account_id    : {df['account_id'].min()} – {df['account_id'].max()}")
print(f"   is_fraud vals : {sorted(df['is_fraud'].unique())}")
print(f"   amount_zscore : {df['amount_zscore'].min():.2f} – {df['amount_zscore'].max():.2f}")

print("\n🧮 Final dtypes of engineered columns:")
print(df[key_cols].dtypes)


📐 Shape     : 284,807 rows × 38 columns

📋 All columns:
    1. transaction_id
    2. Time
    3. V1
    4. V2
    5. V3
    6. V4
    7. V5
    8. V6
    9. V7
   10. V8
   11. V9
   12. V10
   13. V11
   14. V12
   15. V13
   16. V14
   17. V15
   18. V16
   19. V17
   20. V18
   21. V19
   22. V20
   23. V21
   24. V22
   25. V23
   26. V24
   27. V25
   28. V26
   29. V27
   30. V28
   31. Amount
   32. is_fraud
   33. hour_of_day
   34. day_number
   35. unusual_hour
   36. amount_log
   37. amount_zscore
   38. account_id

🔍 Null check on key engineered columns:
transaction_id    0
hour_of_day       0
day_number        0
unusual_hour      0
amount_log        0
amount_zscore     0
account_id        0
dtype: int64

📊 Value ranges:
   hour_of_day   : 0 – 23
   day_number    : 1 – 2
   account_id    : 1 – 1998
   is_fraud vals : [np.int64(0), np.int64(1)]
   amount_zscore : -0.35 – 102.36

🧮 Final dtypes of engineered columns:
transaction_id      int64
hour_of_day         int64
day_n

In [61]:
#  Push the enriched DataFrame into PostgreSQL.

# Why DTYPE_MAP? Without it, SQLAlchemy guesses column types — often wrong (e.g., storing integers as TEXT). By specifying types explicitly:
# • SmallInteger() → for 0/1 flags and small numbers (saves space)
# • Numeric(12, 6) → for decimals with 6 decimal places (Amount, V1–V28)
# • BigInteger() → for transaction_id (284,807 fits in regular Integer but BigInteger is safer)

# Key parameters in to_sql():
# • if_exists='replace' → drops and recreates table every run
# • chunksize=1000 → inserts 1,000 rows per batch (avoids memory overload)
# • method='multi' → one INSERT per batch (faster than row-by-row)
# ── Explicit type map for clean PostgreSQL DDL ────────────
DTYPE_MAP = {
    'transaction_id'  : BigInteger(),
    'Time'            : Numeric(12, 2),
    'Amount'          : Numeric(12, 6),
    'is_fraud'        : SmallInteger(),
    'hour_of_day'     : SmallInteger(),
    'day_number'      : SmallInteger(),
    'unusual_hour'    : SmallInteger(),
    'amount_log'      : Numeric(10, 6),
    'amount_zscore'   : Numeric(10, 6),
    'account_id'      : Integer(),
}
# V1-V28: all get Numeric(12,6)
for col in [f'V{i}' for i in range(1, 29)]:
    DTYPE_MAP[col] = Numeric(12, 6)

# Build connection string
CONN_STR = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
engine = create_engine(CONN_STR)

print("Connecting to PostgreSQL...")
print(f"Loading {len(df):,} rows into table '{TABLE_NAME}'...")

# Push DataFrame to PostgreSQL
df.to_sql(
    name      = TABLE_NAME,
    con       = engine,
    if_exists = 'replace',   # drop + recreate table
    index     = False,        # don't write DataFrame index as a column
    dtype     = DTYPE_MAP,
    chunksize = 1000,         # insert 1000 rows per batch
    method    = 'multi',      # one INSERT per batch
)

# Verify count in DB matches DataFrame
with engine.connect() as conn:
    row_count = conn.execute(
        text(f'SELECT COUNT(*) FROM "{TABLE_NAME}"')
    ).scalar()

print(f"\n✅ Table '{TABLE_NAME}' created in '{DB_NAME}'")
print(f"   Rows in DB  : {row_count:,}")
print(f"   Rows in df  : {len(df):,}")
print(f"   Match       : {row_count == len(df)}")

Connecting to PostgreSQL...
Loading 284,807 rows into table 'transactions'...



✅ Table 'transactions' created in 'fraud_detection'
   Rows in DB  : 284,807
   Rows in df  : 284,807
   Match       : True
